# Advanced Rating Experiments

The purpose of this notebook is to continue from the already completed test prediction stage and run additional experiments in a controlled and reproducible way.

In [ ]:
# Imports

from pathlib import Path
import os
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    mean_absolute_error,
    confusion_matrix,
    classification_report,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    set_seed,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 220)

In [ ]:
# Project configuration
PROJECT_DIR = Path("PATH").resolve()

# Preprocessed datasets
TRAIN_PATH = PROJECT_DIR / "preprocessed_train.csv"
VAL_PATH   = PROJECT_DIR / "preprocessed_val.csv"
TEST_PATH  = PROJECT_DIR / "preprocessed_test.csv"

# Prediction files of the two chosen models
ROBERTA_PRED_PATH = PROJECT_DIR / "final_test_predictions" / "test_predictions_roberta_metadata.csv"
BERT_PRED_PATH    = PROJECT_DIR / "final_test_predictions" / "test_predictions_bert_base.csv"

# Final-output templates
ROBERTA_BASE_OUTPUT_PATH = PROJECT_DIR / "final_output_base_files" / "alternative_1_roberta_metadata_base.csv"
BERT_BASE_OUTPUT_PATH    = PROJECT_DIR / "final_output_base_files" / "alternative_2_bert_base_base.csv"

# Aggregate evaluation files
METRICS_PATH = PROJECT_DIR / "final_test_predictions" / "selected_models_test_metrics.csv"

ROBERTA_REPORT_PATH = PROJECT_DIR / "final_test_predictions" / "classification_report_roberta_metadata_test.csv"
BERT_REPORT_PATH    = PROJECT_DIR / "final_test_predictions" / "classification_report_bert_base_test.csv"

ROBERTA_CM_PATH = PROJECT_DIR / "final_test_predictions" / "confusion_matrix_roberta_metadata_test.csv"
BERT_CM_PATH    = PROJECT_DIR / "final_test_predictions" / "confusion_matrix_bert_base_test.csv"

# Output folder for all new advanced experiments
OUTPUT_DIR = PROJECT_DIR / "advanced_experiments_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# Dataset loading

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"train_df: loaded {train_df.shape[0]:,} rows and {train_df.shape[1]:,} columns")
print(f"val_df:   loaded {val_df.shape[0]:,} rows and {val_df.shape[1]:,} columns")
print(f"test_df:  loaded {test_df.shape[0]:,} rows and {test_df.shape[1]:,} columns")

# Make sure rating is integer
train_df["rating"] = train_df["rating"].astype(int)
val_df["rating"] = val_df["rating"].astype(int)
test_df["rating"] = test_df["rating"].astype(int)

## Experiment 1: Ensemble

The current available prediction files are only for the two best base models:

- `RoBERTa Base + Metadata`
- `BERT Base`


In [ ]:
# Load prediction files

roberta_pred = pd.read_csv(ROBERTA_PRED_PATH)
bert_pred = pd.read_csv(BERT_PRED_PATH)

print("RoBERTa prediction file shape:", roberta_pred.shape)
print("BERT prediction file shape:", bert_pred.shape)

print("\nRoBERTa columns:")
print(list(roberta_pred.columns))

print("\nBERT columns:")
print(list(bert_pred.columns))

display(roberta_pred.head(3))
display(bert_pred.head(3))

In [ ]:
# Validate row alignment

if len(roberta_pred) != len(bert_pred):
    raise ValueError("The two prediction files have different numbers of rows.")

alignment_checks = {
    "uniqueID": (roberta_pred["uniqueID"].astype(str).values == bert_pred["uniqueID"].astype(str).values).all(),
    "true_rating": (roberta_pred["true_rating"].astype(int).values == bert_pred["true_rating"].astype(int).values).all(),
    "review": (roberta_pred["review"].astype(str).values == bert_pred["review"].astype(str).values).all(),
}

print("Alignment checks:")
for key, value in alignment_checks.items():
    print(f"- {key}: {value}")

if not all(alignment_checks.values()):
    raise ValueError("The prediction files are not aligned. Do not run ensemble before fixing row alignment.")

print("\nThe two prediction files are aligned and safe to ensemble.")

In [ ]:
# Evaluation helper functions
#
# The helper functions below compute the same metrics used in the rest of the project:
# - Accuracy / exact-match rate
# - Macro precision / recall / F1
# - Weighted F1
# - MAE
# - Within-1 and within-2 rating accuracy
# - Large error rate: absolute error >= 3

RATING_LABELS = list(range(1, 11))

def clip_rating(pred):
    """Clip predictions to the valid rating range [1, 10]."""
    return np.clip(np.asarray(pred), 1, 10).astype(int)

def round_half_up(x):
    """
    Round values using the conventional half-up rule.

    Python's built-in round uses bankers rounding in some cases.
    For ratings, a clear half-up rule is easier to explain:
        8.5 -> 9
        7.5 -> 8
    """
    return np.floor(np.asarray(x, dtype=float) + 0.5).astype(int)

def compute_rating_metrics(y_true, y_pred, model_name: str) -> dict:
    """Compute all rating metrics for one set of predictions."""
    y_true = np.asarray(y_true).astype(int)
    y_pred = clip_rating(y_pred)

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=RATING_LABELS,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=RATING_LABELS,
        average="weighted",
        zero_division=0,
    )[2]

    acc = accuracy_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    abs_error = np.abs(y_true - y_pred)

    return {
        "model": model_name,
        "accuracy": acc,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "mae": mae,
        "exact_match_rate": acc,
        "within_1_rating": np.mean(abs_error <= 1),
        "within_2_ratings": np.mean(abs_error <= 2),
        "large_error_rate_3_or_more": np.mean(abs_error >= 3),
    }

def make_confusion_matrix_df(y_true, y_pred) -> pd.DataFrame:
    """Create a readable 10-class confusion matrix DataFrame."""
    cm = confusion_matrix(y_true, y_pred, labels=RATING_LABELS)
    return pd.DataFrame(
        cm,
        index=[f"true_{i}" for i in RATING_LABELS],
        columns=[f"pred_{i}" for i in RATING_LABELS],
    )

In [ ]:
# Recompute individual model metrics from row-level files

y_true = roberta_pred["true_rating"].astype(int).values

roberta_y_pred = roberta_pred["predicted_rating"].astype(int).values
bert_y_pred = bert_pred["predicted_rating"].astype(int).values

individual_metrics = pd.DataFrame([
    compute_rating_metrics(y_true, roberta_y_pred, "RoBERTa Base + Metadata"),
    compute_rating_metrics(y_true, bert_y_pred, "BERT Base"),
])

display(individual_metrics)

individual_metrics.to_csv(OUTPUT_DIR / "recomputed_individual_model_metrics.csv", index=False)
print("Saved:", OUTPUT_DIR / "recomputed_individual_model_metrics.csv")

In [ ]:
# Analyze agreement and disagreement between the two models

comparison_df = pd.DataFrame({
    "uniqueID": roberta_pred["uniqueID"],
    "drugName": roberta_pred["drugName"],
    "condition": roberta_pred["condition"],
    "review": roberta_pred["review"],
    "true_rating": y_true,
    "roberta_predicted_rating": roberta_y_pred,
    "bert_predicted_rating": bert_y_pred,
    "roberta_confidence": roberta_pred["prediction_confidence"].astype(float),
    "bert_confidence": bert_pred["prediction_confidence"].astype(float),
})

comparison_df["models_agree"] = (
    comparison_df["roberta_predicted_rating"] == comparison_df["bert_predicted_rating"]
)

comparison_df["prediction_gap"] = (
    comparison_df["roberta_predicted_rating"] - comparison_df["bert_predicted_rating"]
).abs()

comparison_df["roberta_absolute_error"] = (
    comparison_df["true_rating"] - comparison_df["roberta_predicted_rating"]
).abs()

comparison_df["bert_absolute_error"] = (
    comparison_df["true_rating"] - comparison_df["bert_predicted_rating"]
).abs()

agreement_rate = comparison_df["models_agree"].mean()
print(f"Agreement rate: {agreement_rate:.4f}")
print(f"Disagreement count: {(~comparison_df['models_agree']).sum():,}")

print("\nPrediction gap distribution:")
display(comparison_df["prediction_gap"].value_counts().sort_index().rename_axis("absolute_prediction_gap").reset_index(name="count"))

comparison_df.to_csv(OUTPUT_DIR / "two_model_row_level_comparison.csv", index=False)
print("Saved:", OUTPUT_DIR / "two_model_row_level_comparison.csv")

In [ ]:
# Two-model ensemble strategies
#
# Important methodological note:
# - The main assignment already selected RoBERTa+Metadata and BERT Base as
#   the two final alternatives.
# - The ensemble below is an additional experiment.
# - Since only hard predicted ratings and confidence scores are available,
#   it is not possible to average logits/probabilities unless rerunning model inference
#   with probability saving enabled.
#
# Ensemble strategies implemented here:
#
# 1. simple_mean_round_half_up
#    Average the two predicted ratings and round to an integer.
#
# 2. validation_mae_weighted_mean
#    Weighted average using validation MAE from the previous model-selection stage.
#    This avoids choosing weights based on the test set.
#
# 3. confidence_winner
#    If the models disagree, choose the prediction from the model with higher
#    row-level prediction confidence.
#
# 4. confidence_weighted_mean
#    Average the two ratings using their row-level confidence scores as weights.

# Validation MAE values from the previous validation-stage comparison
VALIDATION_MAE = {
    "RoBERTa Base + Metadata": 0.872,
    "BERT Base": 0.884,
}

roberta_conf = comparison_df["roberta_confidence"].astype(float).values
bert_conf = comparison_df["bert_confidence"].astype(float).values

# Strategy 1: simple mean
simple_mean_pred = round_half_up((roberta_y_pred + bert_y_pred) / 2.0)

# Strategy 2: validation-MAE-weighted mean, where lower validation MAE receives higher weight
w_roberta = 1.0 / VALIDATION_MAE["RoBERTa Base + Metadata"]
w_bert = 1.0 / VALIDATION_MAE["BERT Base"]

validation_mae_weighted_pred = round_half_up(
    (w_roberta * roberta_y_pred + w_bert * bert_y_pred) / (w_roberta + w_bert)
)

# Strategy 3: confidence winner
confidence_winner_pred = np.where(
    bert_conf >= roberta_conf,
    bert_y_pred,
    roberta_y_pred,
)

# Strategy 4: confidence-weighted mean, add a tiny epsilon to avoid division by zero in unexpected cases
eps = 1e-12
confidence_weighted_pred = round_half_up(
    (roberta_conf * roberta_y_pred + bert_conf * bert_y_pred) / (roberta_conf + bert_conf + eps)
)

ensemble_predictions = {
    "RoBERTa Base + Metadata": roberta_y_pred,
    "BERT Base": bert_y_pred,
    "Ensemble - Simple Mean": simple_mean_pred,
    "Ensemble - Validation MAE Weighted Mean": validation_mae_weighted_pred,
    "Ensemble - Confidence Winner": confidence_winner_pred,
    "Ensemble - Confidence Weighted Mean": confidence_weighted_pred,
}

ensemble_metrics = pd.DataFrame([
    compute_rating_metrics(y_true, pred, name)
    for name, pred in ensemble_predictions.items()
]).sort_values(["mae", "macro_f1"], ascending=[True, False]).reset_index(drop=True)

display(ensemble_metrics)

ensemble_metrics.to_csv(OUTPUT_DIR / "ensemble_metrics.csv", index=False)
print("Saved:", OUTPUT_DIR / "ensemble_metrics.csv")

In [ ]:
# Save row-level ensemble predictions

ensemble_output = comparison_df.copy()

ensemble_output["ensemble_simple_mean_predicted_rating"] = clip_rating(simple_mean_pred)
ensemble_output["ensemble_validation_mae_weighted_predicted_rating"] = clip_rating(validation_mae_weighted_pred)
ensemble_output["ensemble_confidence_winner_predicted_rating"] = clip_rating(confidence_winner_pred)
ensemble_output["ensemble_confidence_weighted_predicted_rating"] = clip_rating(confidence_weighted_pred)

for col in [
    "ensemble_simple_mean_predicted_rating",
    "ensemble_validation_mae_weighted_predicted_rating",
    "ensemble_confidence_winner_predicted_rating",
    "ensemble_confidence_weighted_predicted_rating",
]:
    ensemble_output[col.replace("predicted_rating", "absolute_error")] = (
        ensemble_output["true_rating"] - ensemble_output[col]
    ).abs()

ensemble_output_path = OUTPUT_DIR / "ensemble_row_level_predictions.csv"
ensemble_output.to_csv(ensemble_output_path, index=False)

print("Saved:", ensemble_output_path)
display(ensemble_output.head(5))

In [ ]:
# Select the best ensemble according to the primary metric

selection_table = ensemble_metrics.sort_values(
    by=["mae", "macro_f1", "accuracy"],
    ascending=[True, False, False],
).reset_index(drop=True)

best_row = selection_table.iloc[0]
best_model_name = best_row["model"]

print("Best model/ensemble according to MAE-first selection:")
display(best_row.to_frame().T)

# Map readable model names to prediction arrays
best_prediction_array = ensemble_predictions[best_model_name]

# Save the best confusion matrix
best_cm = make_confusion_matrix_df(y_true, best_prediction_array)
best_cm_path = OUTPUT_DIR / "best_ensemble_confusion_matrix.csv"
best_cm.to_csv(best_cm_path)

print("Saved:", best_cm_path)
display(best_cm)

In [ ]:
# Per-rating error analysis for all strategies
#
# This table helps identify whether the ensemble improves mainly on:
# - extreme ratings such as 1 and 10
# - middle ratings such as 4, 5, 6, 7, which are harder because reviews are often mixed

per_rating_rows = []

for strategy_name, pred in ensemble_predictions.items():
    pred = clip_rating(pred)
    abs_error = np.abs(y_true - pred)

    for rating in RATING_LABELS:
        mask = (y_true == rating)
        if mask.sum() == 0:
            continue

        per_rating_rows.append({
            "strategy": strategy_name,
            "true_rating": rating,
            "support": int(mask.sum()),
            "accuracy": float(np.mean(pred[mask] == y_true[mask])),
            "mae": float(np.mean(abs_error[mask])),
            "within_1_rating": float(np.mean(abs_error[mask] <= 1)),
            "large_error_rate_3_or_more": float(np.mean(abs_error[mask] >= 3)),
        })

per_rating_metrics = pd.DataFrame(per_rating_rows)

per_rating_path = OUTPUT_DIR / "per_rating_metrics_all_strategies.csv"
per_rating_metrics.to_csv(per_rating_path, index=False)

print("Saved:", per_rating_path)
display(per_rating_metrics.head(20))

In [ ]:
# Focused analysis for borderline ratings 4 to 7

borderline_mask = np.isin(y_true, [4, 5, 6, 7])

borderline_rows = []
for strategy_name, pred in ensemble_predictions.items():
    pred = clip_rating(pred)
    abs_error = np.abs(y_true - pred)

    borderline_rows.append({
        "strategy": strategy_name,
        "support": int(borderline_mask.sum()),
        "accuracy": float(np.mean(pred[borderline_mask] == y_true[borderline_mask])),
        "mae": float(np.mean(abs_error[borderline_mask])),
        "within_1_rating": float(np.mean(abs_error[borderline_mask] <= 1)),
        "within_2_ratings": float(np.mean(abs_error[borderline_mask] <= 2)),
        "large_error_rate_3_or_more": float(np.mean(abs_error[borderline_mask] >= 3)),
    })

borderline_metrics = pd.DataFrame(borderline_rows).sort_values("mae").reset_index(drop=True)

borderline_metrics_path = OUTPUT_DIR / "borderline_ratings_4_to_7_metrics.csv"
borderline_metrics.to_csv(borderline_metrics_path, index=False)

print("Saved:", borderline_metrics_path)
display(borderline_metrics)

In [ ]:
# Optional - Create a deliverable-style base file for the best ensemble

ensemble_base = pd.DataFrame({
    "drug name": roberta_pred["drugName"],
    "condition": roberta_pred["condition"],
    "review": roberta_pred["review"],
    "rating": roberta_pred["true_rating"].astype(int),
    "predicted rating": clip_rating(best_prediction_array),
    "generated summary": np.nan,
    "identified sentiment": np.nan,
})

ensemble_base_path = OUTPUT_DIR / "alternative_3_best_ensemble_base.csv"
ensemble_base.to_csv(ensemble_base_path, index=False)

print("Best strategy used for this file:", best_model_name)
print("Saved:", ensemble_base_path)
display(ensemble_base.head(5))

## Experiment 2: Multi-head rating + sentiment model

The multi-head uses one shared Transformer backbone and two classification heads:

- Head 1 predicts the exact rating from 1 to 10.
- Head 2 predicts a coarse 3-class sentiment label:
  - ratings 1-4 -> negative
  - ratings 5-6 -> neutral
  - ratings 7-10 -> positive

The total loss is:

```text
total_loss = rating_loss + lambda_sentiment * sentiment_loss
```

This can help because the auxiliary sentiment task teaches the model the broad direction of the review while the rating head still learns the exact 1-10 score.

In [ ]:
# Training hyperparameters for the experiments
BACKBONE_MODEL_NAME_OR_PATH = "BERT_Base/checkpoint-16130"

OPTIONAL_MAX_LENGTH = 256
OPTIONAL_BATCH_SIZE = 16
OPTIONAL_EPOCHS = 3
OPTIONAL_LEARNING_RATE = 2e-5
OPTIONAL_RANDOM_SEED = 42

In [ ]:
# Text column for BERT Base checkpoint
# The BERT Base model was trained as a review-only model, so do not use metadata text here

if "model_text_review_only" in train_df.columns:
    TEXT_COLUMN = "model_text_review_only"
elif "review_clean" in train_df.columns:
    TEXT_COLUMN = "review_clean"
else:
    TEXT_COLUMN = "review"

print("Using text column:", TEXT_COLUMN)
print(train_df[TEXT_COLUMN].iloc[0][:300])

In [ ]:
# Multi-head experiment configuration
set_seed(OPTIONAL_RANDOM_SEED)

MULTIHEAD_DIR = OUTPUT_DIR / "multihead_bert_checkpoint_16130"
MULTIHEAD_DIR.mkdir(parents=True, exist_ok=True)

# Warm-start from the selected BERT Base checkpoint
BACKBONE_MODEL_NAME_OR_PATH = str(PROJECT_DIR / "BERT_Base" / "checkpoint-16130")

print("Multi-head experiment configuration\n")
print(f"Backbone checkpoint: {BACKBONE_MODEL_NAME_OR_PATH}")
print(f"Text column: {TEXT_COLUMN}")
print(f"Output directory: {MULTIHEAD_DIR}")
print(f"Max length: {OPTIONAL_MAX_LENGTH}")
print(f"Batch size: {OPTIONAL_BATCH_SIZE}")
print(f"Epochs: {OPTIONAL_EPOCHS}")
print(f"Learning rate: {OPTIONAL_LEARNING_RATE}")
print(f"Random seed: {OPTIONAL_RANDOM_SEED}")

In [ ]:
# Label conversion utilities

def rating_to_10class_label(rating):
    """
    Convert rating from 1-10 into class index 0-9.
    Example:
        rating 1  -> label 0
        rating 10 -> label 9
    """
    return int(rating) - 1


def rating_to_3class_sentiment_label(rating):
    """
    Convert rating into paper-style 3-class sentiment.

    Label mapping:
        0 = negative: ratings 1-4
        1 = neutral:  ratings 5-6
        2 = positive: ratings 7-10
    """
    rating = int(rating)

    if rating <= 4:
        return 0
    elif rating <= 6:
        return 1
    else:
        return 2


SENTIMENT_ID_TO_NAME = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

In [ ]:
# Dataset class for the multi-head model

class MultiHeadDataset(Dataset):
    """
    Dataset for the multi-head model.

    Each sample contains:
    - tokenized text
    - rating label for the 10-class rating head
    - sentiment label for the 3-class auxiliary sentiment head
    """

    def __init__(self, df, tokenizer, text_col, max_length):
        self.df = df.reset_index(drop=True).copy()
        self.texts = self.df[text_col].astype(str).tolist()

        self.rating_labels = (
            self.df["rating"]
            .apply(rating_to_10class_label)
            .astype(int)
            .tolist()
        )

        self.sentiment_labels = (
            self.df["rating"]
            .apply(rating_to_3class_sentiment_label)
            .astype(int)
            .tolist()
        )

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors=None,
        )

        item = {
            key: torch.tensor(value, dtype=torch.long)
            for key, value in encoding.items()
        }

        item["rating_labels"] = torch.tensor(
            self.rating_labels[idx],
            dtype=torch.long,
        )

        item["sentiment_labels"] = torch.tensor(
            self.sentiment_labels[idx],
            dtype=torch.long,
        )

        return item

In [ ]:
# Multi-head Transformer model

class MultiHeadTransformer(nn.Module):
    """
    Shared Transformer encoder with two prediction heads.

    The backbone is initialized from the selected BERT checkpoint.

    Heads:
    - rating_head: predicts 10 rating classes.
    - sentiment_head: predicts 3 sentiment classes.

    Training loss:
        total_loss = rating_loss + lambda_sentiment * sentiment_loss

    Backpropagation is joint:
    - The shared BERT encoder is updated using both losses.
    - The rating head is updated using rating loss.
    - The sentiment head is updated using sentiment loss.
    """

    def __init__(self, model_name_or_path, lambda_sentiment=0.5, dropout=0.1):
        super().__init__()

        self.backbone = AutoModel.from_pretrained(model_name_or_path)
        hidden_size = self.backbone.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.rating_head = nn.Linear(hidden_size, 10)
        self.sentiment_head = nn.Linear(hidden_size, 3)

        self.lambda_sentiment = lambda_sentiment
        self.ce = nn.CrossEntropyLoss()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        rating_labels=None,
        sentiment_labels=None,
        **kwargs,
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # For BERT, the first token is the [CLS] representation
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        cls_embedding = self.dropout(cls_embedding)

        rating_logits = self.rating_head(cls_embedding)
        sentiment_logits = self.sentiment_head(cls_embedding)

        loss = None

        if rating_labels is not None and sentiment_labels is not None:
            rating_loss = self.ce(rating_logits, rating_labels)
            sentiment_loss = self.ce(sentiment_logits, sentiment_labels)

            # Joint multi-task loss - rating is the main task, sentiment is an auxiliary task
            loss = rating_loss + self.lambda_sentiment * sentiment_loss

        return {
            "loss": loss,
            "rating_logits": rating_logits,
            "sentiment_logits": sentiment_logits,
        }

In [ ]:
# Validation metrics for Trainer

def extract_rating_and_sentiment_logits(predictions):
    """
    Trainer may return predictions as a tuple because the model returns
    both rating_logits and sentiment_logits.
    """
    if isinstance(predictions, tuple):
        rating_logits = predictions[0]
        sentiment_logits = predictions[1]
    else:
        rating_logits = predictions
        sentiment_logits = None

    return rating_logits, sentiment_logits


def extract_rating_and_sentiment_labels(label_ids):
    """
    Trainer returns label_ids as a tuple when multiple label columns exist.
    """
    if isinstance(label_ids, tuple):
        rating_labels = label_ids[0]
        sentiment_labels = label_ids[1]
    else:
        rating_labels = label_ids
        sentiment_labels = None

    return rating_labels, sentiment_labels


def compute_multihead_metrics(eval_pred):
    """
    Compute validation metrics at the end of each epoch.

    Main task metrics:
    - rating accuracy
    - rating macro-F1
    - rating MAE
    - within-1 accuracy
    - large error rate

    Auxiliary task metrics:
    - sentiment accuracy
    - sentiment macro-F1
    """
    predictions, label_ids = eval_pred

    rating_logits, sentiment_logits = extract_rating_and_sentiment_logits(predictions)
    rating_labels, sentiment_labels = extract_rating_and_sentiment_labels(label_ids)

    y_true_rating = rating_labels.astype(int) + 1
    y_pred_rating = np.argmax(rating_logits, axis=1) + 1

    abs_errors = np.abs(y_true_rating - y_pred_rating)

    metrics = {
        "rating_accuracy": accuracy_score(y_true_rating, y_pred_rating),
        "rating_macro_f1": f1_score(y_true_rating, y_pred_rating, average="macro"),
        "rating_weighted_f1": f1_score(y_true_rating, y_pred_rating, average="weighted"),
        "rating_mae": mean_absolute_error(y_true_rating, y_pred_rating),
        "rating_within_1": np.mean(abs_errors <= 1),
        "rating_within_2": np.mean(abs_errors <= 2),
        "rating_large_error_ge_3": np.mean(abs_errors >= 3),
    }

    if sentiment_logits is not None and sentiment_labels is not None:
        y_true_sentiment = sentiment_labels.astype(int)
        y_pred_sentiment = np.argmax(sentiment_logits, axis=1)

        metrics.update({
            "sentiment_accuracy": accuracy_score(y_true_sentiment, y_pred_sentiment),
            "sentiment_macro_f1": f1_score(y_true_sentiment, y_pred_sentiment, average="macro"),
            "sentiment_weighted_f1": f1_score(y_true_sentiment, y_pred_sentiment, average="weighted"),
        })

    return metrics

In [ ]:
# Create tokenizer, datasets, and model

tokenizer = AutoTokenizer.from_pretrained(BACKBONE_MODEL_NAME_OR_PATH)

train_dataset = MultiHeadDataset(
    train_df,
    tokenizer,
    TEXT_COLUMN,
    OPTIONAL_MAX_LENGTH,
)

val_dataset = MultiHeadDataset(
    val_df,
    tokenizer,
    TEXT_COLUMN,
    OPTIONAL_MAX_LENGTH,
)

test_dataset = MultiHeadDataset(
    test_df,
    tokenizer,
    TEXT_COLUMN,
    OPTIONAL_MAX_LENGTH,
)

model = MultiHeadTransformer(
    model_name_or_path=BACKBONE_MODEL_NAME_OR_PATH,
    lambda_sentiment=0.5,
    dropout=0.1,
)

print("Tokenizer, datasets, and model were created successfully.")
print(f"Train examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")
print(f"Test examples: {len(test_dataset)}")

In [ ]:
# Training setup
# Main settings:
# - Training loss: every 100 steps
# - Validation metrics: every 1500 steps
# - Checkpoints: every 1500 steps
# - Best model: selected by validation rating MAE

class PrintLogsCallback(TrainerCallback):
    """
    Print important logs clearly during training.

    In some notebook environments, HuggingFace Trainer shows only a compact
    progress table. This callback forces the important metrics to be printed
    whenever they are logged.
    """

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        step = state.global_step
        epoch = logs.get("epoch", None)

        msg = f"[step {step}"
        if epoch is not None:
            msg += f", epoch {epoch:.2f}"
        msg += "] "

        keys_to_print = [
            "loss",
            "eval_loss",
            "eval_rating_mae",
            "eval_rating_accuracy",
            "eval_rating_macro_f1",
            "eval_rating_weighted_f1",
            "eval_rating_within_1",
            "eval_rating_within_2",
            "eval_rating_large_error_ge_3",
            "eval_sentiment_accuracy",
            "eval_sentiment_macro_f1",
            "learning_rate",
        ]

        parts = []
        for key in keys_to_print:
            if key in logs:
                value = logs[key]
                if isinstance(value, float):
                    if key == "learning_rate":
                        parts.append(f"{key}={value:.2e}")
                    else:    
                        parts.append(f"{key}={value:.4f}")
                else:
                    parts.append(f"{key}={value}")

        if parts:
            print(msg + " | ".join(parts), flush=True)


training_args = TrainingArguments(
    output_dir=str(MULTIHEAD_DIR / "checkpoints"),

    learning_rate=OPTIONAL_LEARNING_RATE,
    per_device_train_batch_size=OPTIONAL_BATCH_SIZE,
    per_device_eval_batch_size=OPTIONAL_BATCH_SIZE * 2,
    num_train_epochs=OPTIONAL_EPOCHS,

    # Evaluate every 1500 steps
    eval_strategy="steps",
    eval_steps=1500,

    # Save checkpoints at the same frequency as evaluation
    save_strategy="steps",
    save_steps=1500,

    # Log training loss every 1000 optimization steps
    logging_strategy="steps",
    logging_steps=1000,

    # Select the best checkpoint according to the main rating task, lower MAE is better
    load_best_model_at_end=True,
    metric_for_best_model="eval_rating_mae",
    greater_is_better=False,

    # Keep only two checkpoints to avoid wasting disk space
    save_total_limit=2,

    # Disable external reporting tools
    report_to=[],

    # Use mixed precision on GPU
    fp16=torch.cuda.is_available(),

    # Required because the model uses two custom labels
    label_names=["rating_labels", "sentiment_labels"],
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_multihead_metrics,
    callbacks=[PrintLogsCallback()],
)

print("Trainer is ready")

In [ ]:
# Train the multi-head model

print("Starting multi-head training...")

train_result = trainer.train()

print("\nTraining completed")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best validation metric: {trainer.state.best_metric}")

# Save the best loaded model and tokenizer
trainer.save_model(str(MULTIHEAD_DIR / "best_model"))
tokenizer.save_pretrained(str(MULTIHEAD_DIR / "best_model"))

print(f"Best model saved to: {MULTIHEAD_DIR / 'best_model'}")

In [ ]:
# Save training history

history = pd.DataFrame(trainer.state.log_history)

history_path = MULTIHEAD_DIR / "training_log_history.csv"
history.to_csv(history_path, index=False)

print(f"Training history saved to: {history_path}")
display(history.tail(20))

In [ ]:
# Plot 1: Training loss vs validation loss

plt.figure(figsize=(8, 5))

if "loss" in history.columns:
    train_loss_df = history.dropna(subset=["loss"])
    if not train_loss_df.empty:
        plt.plot(
            train_loss_df["step"],
            train_loss_df["loss"],
            label="Training loss",
        )

if "eval_loss" in history.columns:
    eval_loss_df = history.dropna(subset=["eval_loss"])
    if not eval_loss_df.empty:
        plt.plot(
            eval_loss_df["step"],
            eval_loss_df["eval_loss"],
            marker="o",
            label="Validation loss",
        )

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Multi-head Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

loss_plot_path = MULTIHEAD_DIR / "training_validation_loss.png"
plt.savefig(loss_plot_path, dpi=150)
plt.show()

print(f"Loss curve saved to: {loss_plot_path}")

In [ ]:
# Plot 2: Validation rating MAE
# This is the most important training graph for this task.
# Lower MAE means the predicted rating is closer to the true rating.

if "eval_rating_mae" in history.columns:
    eval_metric_df = history.dropna(subset=["eval_rating_mae"])

    if not eval_metric_df.empty:
        plt.figure(figsize=(8, 5))
        plt.plot(
            eval_metric_df["step"],
            eval_metric_df["eval_rating_mae"],
            marker="o",
            label="Validation rating MAE",
        )

        plt.xlabel("Training step")
        plt.ylabel("MAE")
        plt.title("Multi-head Validation Rating MAE")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        mae_plot_path = MULTIHEAD_DIR / "validation_rating_mae.png"
        plt.savefig(mae_plot_path, dpi=150)
        plt.show()

        print(f"Validation MAE curve saved to: {mae_plot_path}")


# Plot 3: Validation rating accuracy and macro-F1
# This graph checks whether the model improves in exact classification, not only in distance-based error.

metric_cols = ["eval_rating_accuracy", "eval_rating_macro_f1"]
available_cols = [col for col in metric_cols if col in history.columns]

if available_cols:
    eval_rating_df = history.dropna(subset=available_cols, how="all")

    if not eval_rating_df.empty:
        plt.figure(figsize=(8, 5))

        for col in available_cols:
            plt.plot(
                eval_rating_df["step"],
                eval_rating_df[col],
                marker="o",
                label=col,
            )

        plt.xlabel("Training step")
        plt.ylabel("Score")
        plt.title("Multi-head Validation Rating Accuracy and Macro-F1")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        rating_metrics_plot_path = MULTIHEAD_DIR / "validation_rating_accuracy_macro_f1.png"
        plt.savefig(rating_metrics_plot_path, dpi=150)
        plt.show()

        print(f"Validation rating metrics curve saved to: {rating_metrics_plot_path}")

In [ ]:
# Plot 4: Validation sentiment accuracy and macro-F1
# This graph checks whether the auxiliary sentiment task is being learned.
# If sentiment metrics are very poor, the auxiliary task may not be helping.

metric_cols = ["eval_sentiment_accuracy", "eval_sentiment_macro_f1"]
available_cols = [col for col in metric_cols if col in history.columns]

if available_cols:
    eval_sentiment_df = history.dropna(subset=available_cols, how="all")

    if not eval_sentiment_df.empty:
        plt.figure(figsize=(8, 5))

        for col in available_cols:
            plt.plot(
                eval_sentiment_df["step"],
                eval_sentiment_df[col],
                marker="o",
                label=col,
            )

        plt.xlabel("Training step")
        plt.ylabel("Score")
        plt.title("Multi-head Validation Sentiment Accuracy and Macro-F1")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        sentiment_metrics_plot_path = MULTIHEAD_DIR / "validation_sentiment_accuracy_macro_f1.png"
        plt.savefig(sentiment_metrics_plot_path, dpi=150)
        plt.show()

        print(f"Validation sentiment metrics curve saved to: {sentiment_metrics_plot_path}")

In [ ]:
# Final prediction on the test set

print("Running final prediction on the test set...")

pred_output = trainer.predict(test_dataset)

rating_logits, sentiment_logits = extract_rating_and_sentiment_logits(
    pred_output.predictions
)

rating_pred = np.argmax(rating_logits, axis=1) + 1

rating_confidence = torch.softmax(
    torch.tensor(rating_logits),
    dim=1,
).max(dim=1).values.numpy()

true_rating = test_df["rating"].astype(int).values
abs_error = np.abs(true_rating - rating_pred)

if sentiment_logits is not None:
    sentiment_pred_id = np.argmax(sentiment_logits, axis=1)

    sentiment_confidence = torch.softmax(
        torch.tensor(sentiment_logits),
        dim=1,
    ).max(dim=1).values.numpy()

    sentiment_pred = [
        SENTIMENT_ID_TO_NAME[int(x)]
        for x in sentiment_pred_id
    ]
else:
    sentiment_pred_id = np.array([None] * len(test_df))
    sentiment_confidence = np.array([None] * len(test_df))
    sentiment_pred = [None] * len(test_df)

true_sentiment_id = (
    test_df["rating"]
    .apply(rating_to_3class_sentiment_label)
    .astype(int)
    .values
)

true_sentiment = [
    SENTIMENT_ID_TO_NAME[int(x)]
    for x in true_sentiment_id
]

print("Prediction completed")

In [ ]:
# Save final multi-head test metrics

multihead_test_metrics = {
    "model": "Multi-head BERT checkpoint-16130",
    "accuracy": accuracy_score(true_rating, rating_pred),
    "macro_f1": f1_score(true_rating, rating_pred, average="macro"),
    "weighted_f1": f1_score(true_rating, rating_pred, average="weighted"),
    "mae": mean_absolute_error(true_rating, rating_pred),
    "within_1": np.mean(abs_error <= 1),
    "within_2": np.mean(abs_error <= 2),
    "large_error_ge_3": np.mean(abs_error >= 3),
}

if sentiment_logits is not None:
    multihead_test_metrics.update({
        "sentiment_accuracy": accuracy_score(true_sentiment_id, sentiment_pred_id),
        "sentiment_macro_f1": f1_score(true_sentiment_id, sentiment_pred_id, average="macro"),
        "sentiment_weighted_f1": f1_score(true_sentiment_id, sentiment_pred_id, average="weighted"),
    })

multihead_metrics_df = pd.DataFrame([multihead_test_metrics])

metrics_path = MULTIHEAD_DIR / "multihead_test_metrics.csv"
multihead_metrics_df.to_csv(metrics_path, index=False)

print("Final multi-head test metrics:")
display(multihead_metrics_df)

print(f"Saved test metrics to: {metrics_path}")

In [ ]:
# Save row-level predictions
#
# This file is useful for:
# - error analysis
# - comparison to the original BERT / RoBERTa models
# - checking borderline ratings 4-7
# - creating additional ensembles if needed

row_level = test_df.copy()

row_level["multihead_predicted_rating"] = rating_pred
row_level["multihead_rating_confidence"] = rating_confidence
row_level["multihead_absolute_error"] = abs_error

row_level["true_sentiment_3class"] = true_sentiment
row_level["multihead_predicted_sentiment_3class"] = sentiment_pred
row_level["multihead_sentiment_confidence"] = sentiment_confidence

row_level_path = MULTIHEAD_DIR / "multihead_test_row_level_predictions.csv"
row_level.to_csv(row_level_path, index=False)

print(f"Saved row-level predictions to: {row_level_path}")

In [ ]:
# Rating confusion matrix and classification report

rating_cm = pd.DataFrame(
    confusion_matrix(true_rating, rating_pred, labels=list(range(1, 11))),
    index=[f"true_{i}" for i in range(1, 11)],
    columns=[f"pred_{i}" for i in range(1, 11)],
)

rating_cm_path = MULTIHEAD_DIR / "multihead_rating_confusion_matrix.csv"
rating_cm.to_csv(rating_cm_path)

rating_report = pd.DataFrame(
    classification_report(
        true_rating,
        rating_pred,
        labels=list(range(1, 11)),
        output_dict=True,
        zero_division=0,
    )
).transpose()

rating_report_path = MULTIHEAD_DIR / "multihead_rating_classification_report.csv"
rating_report.to_csv(rating_report_path)

print(f"Saved rating confusion matrix to: {rating_cm_path}")
print(f"Saved rating classification report to: {rating_report_path}")


# Sentiment confusion matrix and classification report

if sentiment_logits is not None:
    sentiment_labels_order = [0, 1, 2]
    sentiment_names_order = ["negative", "neutral", "positive"]

    sentiment_cm = pd.DataFrame(
        confusion_matrix(
            true_sentiment_id,
            sentiment_pred_id,
            labels=sentiment_labels_order,
        ),
        index=[f"true_{name}" for name in sentiment_names_order],
        columns=[f"pred_{name}" for name in sentiment_names_order],
    )

    sentiment_cm_path = MULTIHEAD_DIR / "multihead_sentiment_confusion_matrix.csv"
    sentiment_cm.to_csv(sentiment_cm_path)

    sentiment_report = pd.DataFrame(
        classification_report(
            true_sentiment_id,
            sentiment_pred_id,
            labels=sentiment_labels_order,
            target_names=sentiment_names_order,
            output_dict=True,
            zero_division=0,
        )
    ).transpose()

    sentiment_report_path = MULTIHEAD_DIR / "multihead_sentiment_classification_report.csv"
    sentiment_report.to_csv(sentiment_report_path)

    print(f"Saved sentiment confusion matrix to: {sentiment_cm_path}")
    print(f"Saved sentiment classification report to: {sentiment_report_path}")


# Borderline ratings 4-7 analysis

borderline_mask = np.isin(true_rating, [4, 5, 6, 7])

borderline_metrics = {
    "model": "Multi-head BERT checkpoint-16130",
    "subset": "ratings_4_to_7",
    "n_examples": int(borderline_mask.sum()),
    "accuracy": accuracy_score(true_rating[borderline_mask], rating_pred[borderline_mask]),
    "macro_f1": f1_score(true_rating[borderline_mask], rating_pred[borderline_mask], average="macro"),
    "mae": mean_absolute_error(true_rating[borderline_mask], rating_pred[borderline_mask]),
    "within_1": np.mean(abs_error[borderline_mask] <= 1),
    "within_2": np.mean(abs_error[borderline_mask] <= 2),
    "large_error_ge_3": np.mean(abs_error[borderline_mask] >= 3),
}

borderline_metrics_df = pd.DataFrame([borderline_metrics])

borderline_path = MULTIHEAD_DIR / "multihead_borderline_ratings_4_to_7_metrics.csv"
borderline_metrics_df.to_csv(borderline_path, index=False)

print("Borderline ratings 4-7 metrics:")
display(borderline_metrics_df)

print(f"Saved borderline metrics to: {borderline_path}")

In [ ]:
# Create final submission output for Multi-head BERT

FINAL_OUTPUT_DIR = PROJECT_DIR / "final_genai_outputs"

MULTIHEAD_PRED_PATH = (
    MULTIHEAD_DIR / "multihead_test_row_level_predictions.csv"
)

GENAI_REPAIRED_PATH = (
    FINAL_OUTPUT_DIR / "alternative_1_roberta_metadata_final_repaired.csv"
)

FINAL_MULTIHEAD_CSV = FINAL_OUTPUT_DIR / "alternative_2_multihead_bert_final.csv"
FINAL_MULTIHEAD_XLSX = FINAL_OUTPUT_DIR / "alternative_2_multihead_bert_final.xlsx"

multihead_preds = pd.read_csv(MULTIHEAD_PRED_PATH)
genai_output = pd.read_csv(GENAI_REPAIRED_PATH)

print("Loaded files:")
print("Multi-head predictions:", multihead_preds.shape)
print("GenAI repaired output:", genai_output.shape)

In [ ]:
# Build final assignment-style output

final_multihead_output = genai_output[
    [
        "drug name",
        "condition",
        "review",
        "rating",
        "generated summary",
        "identified sentiment",
    ]
].copy()

final_multihead_output.insert(
    loc=4,
    column="predicted rating",
    value=multihead_preds["multihead_predicted_rating"].astype(int),
)

final_multihead_output = final_multihead_output[
    [
        "drug name",
        "condition",
        "review",
        "rating",
        "predicted rating",
        "generated summary",
        "identified sentiment",
    ]
]

display(final_multihead_output.head())
print("Final output shape:", final_multihead_output.shape)

In [ ]:
# Save final Multi-head submission files

final_multihead_output.to_csv(FINAL_MULTIHEAD_CSV, index=False)
final_multihead_output.to_excel(FINAL_MULTIHEAD_XLSX, index=False)

print("Saved final Multi-head output files:")
print(FINAL_MULTIHEAD_CSV)
print(FINAL_MULTIHEAD_XLSX)

## Experiment 3: Soft-label rating model

Soft labels are useful when neighboring ratings are semantically close.

Instead of training only with a hard target such as:

```text
rating 8 -> class 8 has probability 1.0
```

try to train with a smooth target such as:

```text
rating 8 -> class 8 high probability, classes 7 and 9 smaller probability
```

At inference time, the model still returns a single final label:

- either by `argmax`
- or by expected rating followed by rounding

So soft labels affect training, not the final output format.

In [ ]:
# Soft-label experiment configuration

set_seed(OPTIONAL_RANDOM_SEED)

SOFT_LABEL_DIR = OUTPUT_DIR / "soft_labels_bert_checkpoint_16130"
SOFT_LABEL_DIR.mkdir(parents=True, exist_ok=True)

# Warm-start from the already fine-tuned BERT Base model.
SOFT_BACKBONE_MODEL_NAME_OR_PATH = str(PROJECT_DIR / "BERT_Base" / "checkpoint-16130")

# Since this is BERT Base text-only, use review-only text.
if "model_text_review_only" in train_df.columns:
    SOFT_TEXT_COLUMN = "model_text_review_only"
elif "review_clean" in train_df.columns:
    SOFT_TEXT_COLUMN = "review_clean"
else:
    SOFT_TEXT_COLUMN = "review"

SOFT_SIGMA = 1.0

def make_soft_rating_distribution(rating: int, sigma: float = 1.0) -> np.ndarray:
    """
    Create a soft probability distribution over ratings 1-10.

    The true rating receives the highest probability.
    Neighboring ratings receive smaller probabilities.
    Far ratings receive very small probabilities.
    """
    ratings = np.arange(1, 11)
    weights = np.exp(-0.5 * ((ratings - int(rating)) / sigma) ** 2)
    return weights / weights.sum()


# Inspect examples of soft targets.
soft_examples = pd.DataFrame(
    [make_soft_rating_distribution(r, sigma=SOFT_SIGMA) for r in [1, 5, 8, 10]],
    index=["rating_1", "rating_5", "rating_8", "rating_10"],
    columns=[f"class_{i}" for i in range(1, 11)],
)

display(soft_examples.round(4))

In [ ]:
# Soft-label dataset
# Each sample contains:
# - tokenized review text
# - a soft target distribution over 10 rating classes

class SoftLabelDataset(Dataset):
    """Dataset that returns tokenized text and a soft target distribution."""

    def __init__(self, df, tokenizer, text_col, max_length, sigma=1.0):
        self.df = df.reset_index(drop=True).copy()
        self.texts = self.df[text_col].astype(str).tolist()

        self.soft_labels = np.vstack([
            make_soft_rating_distribution(r, sigma=sigma)
            for r in self.df["rating"].astype(int).values
        ]).astype("float32")

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors=None,
        )

        item = {
            key: torch.tensor(value, dtype=torch.long)
            for key, value in encoding.items()
        }

        # Soft labels are probability distributions, so they are float tensors.
        item["labels"] = torch.tensor(self.soft_labels[idx], dtype=torch.float)

        return item

In [ ]:
# Custom trainer and metrics
#
# Normal CrossEntropyLoss expects one hard integer label.
# Here I use KL-divergence between:
# - predicted probability distribution
# - target soft distribution

class SoftLabelTrainer(Trainer):
    """Trainer that optimizes KL-divergence for soft-label training."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        log_probs = F.log_softmax(logits, dim=-1)

        # KLDivLoss expects log-probabilities as input and probabilities as target.
        loss = F.kl_div(log_probs, labels, reduction="batchmean")

        return (loss, outputs) if return_outputs else loss


def round_half_up_array(x):
    """Round values to nearest integer using half-up behavior, then clip to 1-10."""
    return np.clip(np.floor(x + 0.5).astype(int), 1, 10)


def predictions_from_logits(logits):
    """
    Convert logits into two possible rating predictions:
    1. Argmax prediction: most likely class.
    2. Expected-rating prediction: weighted average over rating probabilities.
    """
    logits = np.asarray(logits)
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = probs / probs.sum(axis=1, keepdims=True)

    rating_values = np.arange(1, 11)

    argmax_pred = np.argmax(probs, axis=1) + 1
    expected_continuous = (probs * rating_values).sum(axis=1)
    expected_pred = round_half_up_array(expected_continuous)

    confidence = probs.max(axis=1)

    return probs, argmax_pred, expected_continuous, expected_pred, confidence


def compute_soft_label_metrics(eval_pred):
    """
    Validation metrics during training.

    labels are soft distributions, but their argmax is the original true rating.
    We report both:
    - argmax metrics
    - expected-rating metrics
    """
    logits, labels = eval_pred

    y_true = np.argmax(labels, axis=1) + 1
    _, argmax_pred, _, expected_pred, _ = predictions_from_logits(logits)

    argmax_abs_error = np.abs(y_true - argmax_pred)
    expected_abs_error = np.abs(y_true - expected_pred)

    return {
        "argmax_accuracy": accuracy_score(y_true, argmax_pred),
        "argmax_macro_f1": f1_score(y_true, argmax_pred, average="macro"),
        "argmax_mae": mean_absolute_error(y_true, argmax_pred),
        "argmax_within_1": np.mean(argmax_abs_error <= 1),

        "expected_accuracy": accuracy_score(y_true, expected_pred),
        "expected_macro_f1": f1_score(y_true, expected_pred, average="macro"),
        "expected_mae": mean_absolute_error(y_true, expected_pred),
        "expected_within_1": np.mean(expected_abs_error <= 1),
    }


class PrintLogsCallback(TrainerCallback):
    """Print important logs clearly during training."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        step = state.global_step
        epoch = logs.get("epoch", None)

        msg = f"[step {step}"
        if epoch is not None:
            msg += f", epoch {epoch:.2f}"
        msg += "] "

        keys = [
            "loss",
            "eval_loss",
            "eval_argmax_mae",
            "eval_expected_mae",
            "eval_argmax_accuracy",
            "eval_expected_accuracy",
            "learning_rate",
        ]

        parts = []
        for key in keys:
            if key in logs:
                value = logs[key]
                if isinstance(value, float):
                    if key == "learning_rate":
                        parts.append(f"{key}={value:.2e}")
                    else:
                        parts.append(f"{key}={value:.4f}")
                else:
                    parts.append(f"{key}={value}")

        if parts:
            print(msg + " | ".join(parts), flush=True)

In [ ]:
# Create tokenizer, datasets, model, and trainer

tokenizer = AutoTokenizer.from_pretrained(SOFT_BACKBONE_MODEL_NAME_OR_PATH)

train_soft_dataset = SoftLabelDataset(
    train_df,
    tokenizer,
    SOFT_TEXT_COLUMN,
    OPTIONAL_MAX_LENGTH,
    sigma=SOFT_SIGMA,
)

val_soft_dataset = SoftLabelDataset(
    val_df,
    tokenizer,
    SOFT_TEXT_COLUMN,
    OPTIONAL_MAX_LENGTH,
    sigma=SOFT_SIGMA,
)

test_soft_dataset = SoftLabelDataset(
    test_df,
    tokenizer,
    SOFT_TEXT_COLUMN,
    OPTIONAL_MAX_LENGTH,
    sigma=SOFT_SIGMA,
)

soft_model = AutoModelForSequenceClassification.from_pretrained(
    SOFT_BACKBONE_MODEL_NAME_OR_PATH,
)

training_args = TrainingArguments(
    output_dir=str(SOFT_LABEL_DIR / "checkpoints"),

    learning_rate=OPTIONAL_LEARNING_RATE,
    per_device_train_batch_size=OPTIONAL_BATCH_SIZE,
    per_device_eval_batch_size=OPTIONAL_BATCH_SIZE * 2,
    num_train_epochs=OPTIONAL_EPOCHS,

    # Same monitoring style as the multi-head experiment
    eval_strategy="steps",
    eval_steps=1500,
    save_strategy="steps",
    save_steps=1500,

    logging_strategy="steps",
    logging_steps=1000,

    # Since this experiment is ordinal, select the best model by expected-rating MAE
    load_best_model_at_end=True,
    metric_for_best_model="eval_expected_mae",
    greater_is_better=False,

    save_total_limit=2,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

soft_trainer = SoftLabelTrainer(
    model=soft_model,
    args=training_args,
    train_dataset=train_soft_dataset,
    eval_dataset=val_soft_dataset,
    compute_metrics=compute_soft_label_metrics,
    callbacks=[PrintLogsCallback()],
)

print("Soft-label trainer is ready")

In [ ]:
# Train soft-label model

print("Starting soft-label training...")

soft_train_result = soft_trainer.train()

print("Training completed.")
print("Best checkpoint:", soft_trainer.state.best_model_checkpoint)
print("Best validation metric:", soft_trainer.state.best_metric)

soft_trainer.save_model(str(SOFT_LABEL_DIR / "best_model"))
tokenizer.save_pretrained(str(SOFT_LABEL_DIR / "best_model"))

history = pd.DataFrame(soft_trainer.state.log_history)
history_path = SOFT_LABEL_DIR / "soft_label_training_log_history.csv"
history.to_csv(history_path, index=False)

print("Training history saved to:", history_path)
display(history.tail(20))


# Plot training and validation loss
plt.figure(figsize=(8, 5))

if "loss" in history.columns:
    train_loss_df = history.dropna(subset=["loss"])
    if not train_loss_df.empty:
        plt.plot(train_loss_df["step"], train_loss_df["loss"], label="Training loss")

if "eval_loss" in history.columns:
    eval_loss_df = history.dropna(subset=["eval_loss"])
    if not eval_loss_df.empty:
        plt.plot(eval_loss_df["step"], eval_loss_df["eval_loss"], marker="o", label="Validation loss")

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Soft-label Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

loss_plot_path = SOFT_LABEL_DIR / "soft_label_training_validation_loss.png"
plt.savefig(loss_plot_path, dpi=150)
plt.show()


# Plot validation MAE for both inference methods
metric_cols = ["eval_argmax_mae", "eval_expected_mae"]
available_cols = [col for col in metric_cols if col in history.columns]

if available_cols:
    eval_df = history.dropna(subset=available_cols, how="all")

    plt.figure(figsize=(8, 5))

    for col in available_cols:
        plt.plot(eval_df["step"], eval_df[col], marker="o", label=col)

    plt.xlabel("Training step")
    plt.ylabel("MAE")
    plt.title("Soft-label Validation MAE")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    mae_plot_path = SOFT_LABEL_DIR / "soft_label_validation_mae.png"
    plt.savefig(mae_plot_path, dpi=150)
    plt.show()

print("Plots saved to:", SOFT_LABEL_DIR)

In [ ]:
# Final prediction on test set

print("Running soft-label prediction on test set...")

pred_output = soft_trainer.predict(test_soft_dataset)
logits = pred_output.predictions

probs, argmax_pred, expected_continuous, expected_pred, confidence = predictions_from_logits(logits)

true_rating = test_df["rating"].astype(int).values

argmax_abs_error = np.abs(true_rating - argmax_pred)
expected_abs_error = np.abs(true_rating - expected_pred)

soft_label_metrics = pd.DataFrame([
    {
        "model": "Soft Labels BERT checkpoint-16130 - Argmax",
        "accuracy": accuracy_score(true_rating, argmax_pred),
        "macro_f1": f1_score(true_rating, argmax_pred, average="macro"),
        "weighted_f1": f1_score(true_rating, argmax_pred, average="weighted"),
        "mae": mean_absolute_error(true_rating, argmax_pred),
        "within_1": np.mean(argmax_abs_error <= 1),
        "within_2": np.mean(argmax_abs_error <= 2),
        "large_error_ge_3": np.mean(argmax_abs_error >= 3),
    },
    {
        "model": "Soft Labels BERT checkpoint-16130 - Expected Rating",
        "accuracy": accuracy_score(true_rating, expected_pred),
        "macro_f1": f1_score(true_rating, expected_pred, average="macro"),
        "weighted_f1": f1_score(true_rating, expected_pred, average="weighted"),
        "mae": mean_absolute_error(true_rating, expected_pred),
        "within_1": np.mean(expected_abs_error <= 1),
        "within_2": np.mean(expected_abs_error <= 2),
        "large_error_ge_3": np.mean(expected_abs_error >= 3),
    },
])

metrics_path = SOFT_LABEL_DIR / "soft_label_test_metrics.csv"
soft_label_metrics.to_csv(metrics_path, index=False)

display(soft_label_metrics)
print("Saved metrics to:", metrics_path)


# Save row-level predictions for analysis
soft_row_level = test_df.copy()
soft_row_level["soft_argmax_predicted_rating"] = argmax_pred
soft_row_level["soft_expected_rating_continuous"] = expected_continuous
soft_row_level["soft_expected_predicted_rating"] = expected_pred
soft_row_level["soft_prediction_confidence"] = confidence
soft_row_level["soft_argmax_absolute_error"] = argmax_abs_error
soft_row_level["soft_expected_absolute_error"] = expected_abs_error

row_level_path = SOFT_LABEL_DIR / "soft_label_test_row_level_predictions.csv"
soft_row_level.to_csv(row_level_path, index=False)

print("Saved row-level predictions to:", row_level_path)


# Save confusion matrix and classification report for the expected-rating prediction
soft_cm = pd.DataFrame(
    confusion_matrix(true_rating, expected_pred, labels=list(range(1, 11))),
    index=[f"true_{i}" for i in range(1, 11)],
    columns=[f"pred_{i}" for i in range(1, 11)],
)

cm_path = SOFT_LABEL_DIR / "soft_label_expected_confusion_matrix.csv"
soft_cm.to_csv(cm_path)

soft_report = pd.DataFrame(
    classification_report(
        true_rating,
        expected_pred,
        labels=list(range(1, 11)),
        output_dict=True,
        zero_division=0,
    )
).transpose()

report_path = SOFT_LABEL_DIR / "soft_label_expected_classification_report.csv"
soft_report.to_csv(report_path)

print("Saved confusion matrix to:", cm_path)
print("Saved classification report to:", report_path)

In [ ]:
# Complete Soft-label analysis files from row-level predictions

ROW_LEVEL_PATH = SOFT_LABEL_DIR / "soft_label_test_row_level_predictions.csv"

print("Loading row-level predictions from:")
print(ROW_LEVEL_PATH)

soft_rows = pd.read_csv(ROW_LEVEL_PATH)


# Validate required columns

required_cols = [
    "rating",
    "soft_argmax_predicted_rating",
    "soft_expected_predicted_rating",
]

missing_cols = [col for col in required_cols if col not in soft_rows.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

true_rating = soft_rows["rating"].astype(int).values
argmax_pred = soft_rows["soft_argmax_predicted_rating"].astype(int).values
expected_pred = soft_rows["soft_expected_predicted_rating"].astype(int).values

rating_labels = list(range(1, 11))


# Argmax classification report

argmax_report = pd.DataFrame(
    classification_report(
        true_rating,
        argmax_pred,
        labels=rating_labels,
        output_dict=True,
        zero_division=0,
    )
).transpose()

argmax_report_path = SOFT_LABEL_DIR / "soft_label_argmax_classification_report.csv"
argmax_report.to_csv(argmax_report_path)

print("Saved Argmax classification report:")
print(argmax_report_path)


# Argmax confusion matrix

argmax_cm = pd.DataFrame(
    confusion_matrix(
        true_rating,
        argmax_pred,
        labels=rating_labels,
    ),
    index=[f"true_{i}" for i in rating_labels],
    columns=[f"pred_{i}" for i in rating_labels],
)

argmax_cm_path = SOFT_LABEL_DIR / "soft_label_argmax_confusion_matrix.csv"
argmax_cm.to_csv(argmax_cm_path)

print("Saved Argmax confusion matrix:")
print(argmax_cm_path)


# Borderline ratings 4-7 metrics

def compute_subset_metrics(y_true, y_pred, name, subset_name):
    abs_error = np.abs(y_true - y_pred)

    return {
        "model": name,
        "subset": subset_name,
        "n_examples": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "mae": mean_absolute_error(y_true, y_pred),
        "within_1": np.mean(abs_error <= 1),
        "within_2": np.mean(abs_error <= 2),
        "large_error_ge_3": np.mean(abs_error >= 3),
    }

borderline_mask = np.isin(true_rating, [4, 5, 6, 7])

borderline_metrics = pd.DataFrame([
    compute_subset_metrics(
        true_rating[borderline_mask],
        argmax_pred[borderline_mask],
        "Soft Labels - Argmax",
        "ratings_4_to_7",
    ),
    compute_subset_metrics(
        true_rating[borderline_mask],
        expected_pred[borderline_mask],
        "Soft Labels - Expected Rating",
        "ratings_4_to_7",
    ),
])

borderline_path = SOFT_LABEL_DIR / "soft_label_borderline_ratings_4_to_7_metrics.csv"
borderline_metrics.to_csv(borderline_path, index=False)

print("Saved borderline ratings metrics:")
print(borderline_path)

display(borderline_metrics)

print("\nDone. Additional Soft-label analysis files were created successfully.")

In [ ]:
# Final Soft Labels output creation

PROJECT_DIR = Path("PATH").resolve()

FINAL_OUTPUT_DIR = PROJECT_DIR / "final_genai_outputs"
SOFT_LABEL_DIR = PROJECT_DIR / "advanced_experiments_results" / "soft_labels_bert_checkpoint_16130"

GENAI_REPAIRED_PATH = FINAL_OUTPUT_DIR / "alternative_1_roberta_metadata_final_repaired.csv"
SOFT_LABEL_PRED_PATH = SOFT_LABEL_DIR / "soft_label_test_row_level_predictions.csv"

FINAL_SOFT_CSV = FINAL_OUTPUT_DIR / "alternative_2_soft_labels_argmax_final.csv"
FINAL_SOFT_XLSX = FINAL_OUTPUT_DIR / "alternative_2_soft_labels_argmax_final.xlsx"

genai_output = pd.read_csv(GENAI_REPAIRED_PATH)
soft_preds = pd.read_csv(SOFT_LABEL_PRED_PATH)

print("GenAI output shape:", genai_output.shape)
print("Soft Labels predictions shape:", soft_preds.shape)

display(genai_output.head(2))
display(soft_preds.head(2))

In [ ]:
# Build final output file for Soft Labels Argmax

required_genai_cols = [
    "drug name",
    "condition",
    "review",
    "rating",
    "generated summary",
    "identified sentiment",
]

missing_genai_cols = [col for col in required_genai_cols if col not in genai_output.columns]
if missing_genai_cols:
    raise ValueError(f"Missing columns in GenAI output: {missing_genai_cols}")

if "soft_argmax_predicted_rating" not in soft_preds.columns:
    raise ValueError("Missing column: soft_argmax_predicted_rating")

final_soft_output = genai_output[required_genai_cols].copy()

final_soft_output.insert(
    4,
    "predicted rating",
    soft_preds["soft_argmax_predicted_rating"].astype(int).values,
)

final_soft_output = final_soft_output[
    [
        "drug name",
        "condition",
        "review",
        "rating",
        "predicted rating",
        "generated summary",
        "identified sentiment",
    ]
]

display(final_soft_output.head())
print("Final Soft Labels output shape:", final_soft_output.shape)

In [ ]:
# Save final Soft Labels output

final_soft_output.to_csv(FINAL_SOFT_CSV, index=False)
final_soft_output.to_excel(FINAL_SOFT_XLSX, index=False)

print("Saved final Soft Labels CSV to:")
print(FINAL_SOFT_CSV)

print("\nSaved final Soft Labels Excel to:")
print(FINAL_SOFT_XLSX)

## Experiment 4: CORAL-style ordinal model

CORAL-style ordinal classification changes the prediction target.

Instead of predicting one of 10 independent classes, the model predicts 9 binary threshold decisions:

```text
Is rating > 1?
Is rating > 2?
...
Is rating > 9?
```

A rating of 8 is represented as:

```text
[1, 1, 1, 1, 1, 1, 1, 0, 0]
```

This makes the ordinal structure explicit.

At inference time, the final rating is:

```text
1 + number of thresholds predicted as True
```


In [ ]:
# CORAL-style ordinal experiment configuration

set_seed(OPTIONAL_RANDOM_SEED)

CORAL_DIR = OUTPUT_DIR / "coral_style_bert_checkpoint_16130"
CORAL_DIR.mkdir(parents=True, exist_ok=True)

# Warm-start from the already fine-tuned BERT Base model.
CORAL_BACKBONE_MODEL_NAME_OR_PATH = str(PROJECT_DIR / "BERT_Base" / "checkpoint-16130")

# Since this is a BERT Base text-only checkpoint, use review-only text.
if "model_text_review_only" in train_df.columns:
    CORAL_TEXT_COLUMN = "model_text_review_only"
elif "review_clean" in train_df.columns:
    CORAL_TEXT_COLUMN = "review_clean"
else:
    CORAL_TEXT_COLUMN = "review"

In [ ]:
# CORAL-style threshold label utility
#
# Example: rating = 5
#
# thresholds: rating > 1 : 1, rating > 2 : 1, rating > 3 : 1, rating > 4 : 1
#             rating > 5 : 0 ..... rating > 9 : 0

def rating_to_coral_thresholds(rating: int) -> np.ndarray:
    """
    Convert a rating in 1-10 into 9 binary ordinal threshold labels.

    Output shape:
        (9,)

    Example:
        rating = 1  -> [0,0,0,0,0,0,0,0,0]
        rating = 10 -> [1,1,1,1,1,1,1,1,1]
    """
    rating = int(rating)

    return np.array(
        [1 if rating > threshold else 0 for threshold in range(1, 10)],
        dtype="float32",
    )


coral_examples = pd.DataFrame(
    [rating_to_coral_thresholds(r) for r in [1, 5, 8, 10]],
    index=["rating_1", "rating_5", "rating_8", "rating_10"],
    columns=[f"rating_gt_{i}" for i in range(1, 10)],
)

display(coral_examples)

In [ ]:
# CORAL-style dataset
# Each sample contains:
# - tokenized review text
# - 9 ordinal threshold labels

class CoralDataset(Dataset):
    """Dataset that returns tokenized text and 9 ordinal threshold labels."""

    def __init__(self, df, tokenizer, text_col, max_length):
        self.df = df.reset_index(drop=True).copy()
        self.texts = self.df[text_col].astype(str).tolist()

        self.threshold_labels = np.vstack([
            rating_to_coral_thresholds(r)
            for r in self.df["rating"].astype(int).values
        ]).astype("float32")

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors=None,
        )

        item = {
            key: torch.tensor(value, dtype=torch.long)
            for key, value in encoding.items()
        }

        # Threshold labels are multi-label binary targets, so they are floats
        item["labels"] = torch.tensor(self.threshold_labels[idx], dtype=torch.float)

        return item

In [ ]:
# CORAL-style Transformer model
# This is a simplified CORAL-style model:
# - Shared BERT encoder
# - One linear head with 9 outputs
# - BCEWithLogitsLoss over ordinal thresholds
#
# Note:
# A full CORAL implementation may include additional constraints on the bias terms.

class CoralStyleTransformer(nn.Module):
    """Transformer encoder with 9 ordinal threshold outputs."""

    def __init__(self, model_name_or_path, dropout=0.1):
        super().__init__()

        self.backbone = AutoModel.from_pretrained(model_name_or_path)
        hidden_size = self.backbone.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.threshold_head = nn.Linear(hidden_size, 9)

        # Binary cross entropy is applied independently to each threshold.
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # For BERT, the first token is the [CLS] representation
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        cls_embedding = self.dropout(cls_embedding)

        threshold_logits = self.threshold_head(cls_embedding)

        loss = None
        if labels is not None:
            loss = self.loss_fn(threshold_logits, labels)

        return {
            "loss": loss,
            "logits": threshold_logits,
        }

In [ ]:
# Prediction utilities and validation metrics

def sigmoid_np(x):
    """Numerically stable sigmoid for numpy arrays."""
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))


def coral_logits_to_rating_threshold(logits, threshold=0.5):
    """
    Convert 9 threshold logits into final integer ratings.

    Each sigmoid(logit) is interpreted as P(rating > threshold_i).

    Final rating:
        1 + number of thresholds with probability >= 0.5
    """
    probs = sigmoid_np(logits)
    passed_thresholds = (probs >= threshold).sum(axis=1)

    return np.clip(1 + passed_thresholds, 1, 10).astype(int)


def coral_logits_to_rating_expected(logits):
    """
    Alternative inference method.

    Instead of hard thresholding each ordinal output, compute:
        expected rating = 1 + sum(P(rating > threshold_i))

    Then round to an integer rating.
    """
    probs = sigmoid_np(logits)
    expected_rating = 1.0 + probs.sum(axis=1)

    return np.clip(np.floor(expected_rating + 0.5).astype(int), 1, 10)


def true_rating_from_threshold_labels(labels):
    """
    Recover true rating from threshold labels.

    If rating = 5, four thresholds are passed:
        rating > 1, 2, 3, 4

    Therefore:
        rating = 1 + sum(threshold labels)
    """
    return (1 + labels.sum(axis=1)).astype(int)


def compute_basic_rating_metrics(y_true, y_pred, prefix):
    """Compute standard rating metrics."""
    abs_error = np.abs(y_true - y_pred)

    return {
        f"{prefix}_accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}_macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}_weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        f"{prefix}_mae": mean_absolute_error(y_true, y_pred),
        f"{prefix}_within_1": np.mean(abs_error <= 1),
        f"{prefix}_within_2": np.mean(abs_error <= 2),
        f"{prefix}_large_error_ge_3": np.mean(abs_error >= 3),
    }


def compute_coral_metrics(eval_pred):
    """
    Validation metrics during training.

    We report two prediction methods:
    1. Threshold prediction: standard CORAL-style inference.
    2. Expected prediction: smoother prediction from summed threshold probabilities.
    """
    logits = eval_pred.predictions
    labels = eval_pred.label_ids

    y_true = true_rating_from_threshold_labels(labels)

    threshold_pred = coral_logits_to_rating_threshold(logits, threshold=0.5)
    expected_pred = coral_logits_to_rating_expected(logits)

    metrics = {}
    metrics.update(compute_basic_rating_metrics(y_true, threshold_pred, "threshold"))
    metrics.update(compute_basic_rating_metrics(y_true, expected_pred, "expected"))

    return metrics


class CoralPrintLogsCallback(TrainerCallback):
    """Print important training and validation logs clearly."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        step = state.global_step
        epoch = logs.get("epoch", None)

        msg = f"[step {step}"
        if epoch is not None:
            msg += f", epoch {epoch:.2f}"
        msg += "] "

        keys = [
            "loss",
            "eval_loss",
            "eval_threshold_mae",
            "eval_expected_mae",
            "eval_threshold_accuracy",
            "eval_expected_accuracy",
            "eval_threshold_macro_f1",
            "eval_expected_macro_f1",
            "learning_rate",
        ]

        parts = []
        for key in keys:
            if key in logs:
                value = logs[key]
                if isinstance(value, float):
                    if key == "learning_rate":
                        parts.append(f"{key}={value:.2e}")
                    else:
                        parts.append(f"{key}={value:.4f}")
                else:
                    parts.append(f"{key}={value}")

        if parts:
            print(msg + " | ".join(parts), flush=True)

In [ ]:
# Create tokenizer, datasets, model, and trainer

tokenizer = AutoTokenizer.from_pretrained(CORAL_BACKBONE_MODEL_NAME_OR_PATH)

train_coral_dataset = CoralDataset(
    train_df,
    tokenizer,
    CORAL_TEXT_COLUMN,
    OPTIONAL_MAX_LENGTH,
)

val_coral_dataset = CoralDataset(
    val_df,
    tokenizer,
    CORAL_TEXT_COLUMN,
    OPTIONAL_MAX_LENGTH,
)

test_coral_dataset = CoralDataset(
    test_df,
    tokenizer,
    CORAL_TEXT_COLUMN,
    OPTIONAL_MAX_LENGTH,
)

coral_model = CoralStyleTransformer(
    model_name_or_path=CORAL_BACKBONE_MODEL_NAME_OR_PATH,
    dropout=0.1,
)

training_args = TrainingArguments(
    output_dir=str(CORAL_DIR / "checkpoints"),

    learning_rate=OPTIONAL_LEARNING_RATE,
    per_device_train_batch_size=OPTIONAL_BATCH_SIZE,
    per_device_eval_batch_size=OPTIONAL_BATCH_SIZE * 2,
    num_train_epochs=OPTIONAL_EPOCHS,

    # More detailed validation curves
    eval_strategy="steps",
    eval_steps=1500,
    save_strategy="steps",
    save_steps=1500,

    # Training loss logging
    logging_strategy="steps",
    logging_steps=1000,

    # Standard CORAL-style prediction uses threshold MAE
    load_best_model_at_end=True,
    metric_for_best_model="eval_threshold_mae",
    greater_is_better=False,

    save_total_limit=2,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

coral_trainer = Trainer(
    model=coral_model,
    args=training_args,
    train_dataset=train_coral_dataset,
    eval_dataset=val_coral_dataset,
    compute_metrics=compute_coral_metrics,
    callbacks=[CoralPrintLogsCallback()],
)

print("CORAL-style trainer is ready")

In [ ]:
# Train CORAL-style model

print("Starting CORAL-style training...")

coral_train_result = coral_trainer.train()

print("Training completed.")
print("Best checkpoint:", coral_trainer.state.best_model_checkpoint)
print("Best validation metric:", coral_trainer.state.best_metric)

coral_trainer.save_model(str(CORAL_DIR / "best_model"))
tokenizer.save_pretrained(str(CORAL_DIR / "best_model"))

history = pd.DataFrame(coral_trainer.state.log_history)

history_path = CORAL_DIR / "coral_training_log_history.csv"
history.to_csv(history_path, index=False)

print("Training history saved to:", history_path)
display(history.tail(20))


# Plot training and validation loss

plt.figure(figsize=(8, 5))

if "loss" in history.columns:
    train_loss_df = history.dropna(subset=["loss"])
    if not train_loss_df.empty:
        plt.plot(train_loss_df["step"], train_loss_df["loss"], label="Training loss")

if "eval_loss" in history.columns:
    eval_loss_df = history.dropna(subset=["eval_loss"])
    if not eval_loss_df.empty:
        plt.plot(eval_loss_df["step"], eval_loss_df["eval_loss"], marker="o", label="Validation loss")

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("CORAL-style Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

loss_plot_path = CORAL_DIR / "coral_training_validation_loss.png"
plt.savefig(loss_plot_path, dpi=150)
plt.show()


# Plot validation MAE

metric_cols = ["eval_threshold_mae", "eval_expected_mae"]
available_cols = [col for col in metric_cols if col in history.columns]

if available_cols:
    eval_df = history.dropna(subset=available_cols, how="all")

    plt.figure(figsize=(8, 5))

    for col in available_cols:
        plt.plot(eval_df["step"], eval_df[col], marker="o", label=col)

    plt.xlabel("Training step")
    plt.ylabel("MAE")
    plt.title("CORAL-style Validation MAE")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    mae_plot_path = CORAL_DIR / "coral_validation_mae.png"
    plt.savefig(mae_plot_path, dpi=150)
    plt.show()


# Plot validation accuracy and macro-F1

metric_cols = [
    "eval_threshold_accuracy",
    "eval_expected_accuracy",
    "eval_threshold_macro_f1",
    "eval_expected_macro_f1",
]
available_cols = [col for col in metric_cols if col in history.columns]

if available_cols:
    eval_df = history.dropna(subset=available_cols, how="all")

    plt.figure(figsize=(8, 5))

    for col in available_cols:
        plt.plot(eval_df["step"], eval_df[col], marker="o", label=col)

    plt.xlabel("Training step")
    plt.ylabel("Score")
    plt.title("CORAL-style Validation Accuracy and Macro-F1")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    metrics_plot_path = CORAL_DIR / "coral_validation_accuracy_macro_f1.png"
    plt.savefig(metrics_plot_path, dpi=150)
    plt.show()

print("Plots saved to:", CORAL_DIR)

In [ ]:
# Final prediction on test set

print("Running CORAL-style prediction on test set...")

pred_output = coral_trainer.predict(test_coral_dataset)
coral_logits = pred_output.predictions
coral_probs = sigmoid_np(coral_logits)

threshold_pred = coral_logits_to_rating_threshold(coral_logits, threshold=0.5)
expected_pred = coral_logits_to_rating_expected(coral_logits)

true_rating = test_df["rating"].astype(int).values

threshold_abs_error = np.abs(true_rating - threshold_pred)
expected_abs_error = np.abs(true_rating - expected_pred)

coral_metrics = pd.DataFrame([
    {
        "model": "CORAL-style BERT checkpoint-16130 - Threshold",
        **compute_basic_rating_metrics(true_rating, threshold_pred, "test"),
    },
    {
        "model": "CORAL-style BERT checkpoint-16130 - Expected",
        **compute_basic_rating_metrics(true_rating, expected_pred, "test"),
    },
])

# Clean metric names for readability
coral_metrics = coral_metrics.rename(columns={
    "test_accuracy": "accuracy",
    "test_macro_f1": "macro_f1",
    "test_weighted_f1": "weighted_f1",
    "test_mae": "mae",
    "test_within_1": "within_1",
    "test_within_2": "within_2",
    "test_large_error_ge_3": "large_error_ge_3",
})

metrics_path = CORAL_DIR / "coral_style_test_metrics.csv"
coral_metrics.to_csv(metrics_path, index=False)

display(coral_metrics)
print("Saved metrics to:", metrics_path)


# Save row-level predictions

coral_row_level = test_df.copy()
coral_row_level["coral_threshold_predicted_rating"] = threshold_pred
coral_row_level["coral_expected_predicted_rating"] = expected_pred
coral_row_level["coral_threshold_absolute_error"] = threshold_abs_error
coral_row_level["coral_expected_absolute_error"] = expected_abs_error
coral_row_level["coral_mean_threshold_probability"] = coral_probs.mean(axis=1)
coral_row_level["coral_sum_threshold_probabilities"] = coral_probs.sum(axis=1)

row_level_path = CORAL_DIR / "coral_style_test_row_level_predictions.csv"
coral_row_level.to_csv(row_level_path, index=False)

print("Saved row-level predictions to:", row_level_path)

In [ ]:
# Reports, confusion matrices, and borderline analysis

rating_labels = list(range(1, 11))

def save_report_and_confusion(y_true, y_pred, prefix):
    """Save classification report and confusion matrix."""
    report = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=rating_labels,
            output_dict=True,
            zero_division=0,
        )
    ).transpose()

    report_path = CORAL_DIR / f"{prefix}_classification_report.csv"
    report.to_csv(report_path)

    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=rating_labels),
        index=[f"true_{i}" for i in rating_labels],
        columns=[f"pred_{i}" for i in rating_labels],
    )

    cm_path = CORAL_DIR / f"{prefix}_confusion_matrix.csv"
    cm.to_csv(cm_path)

    print(f"Saved report: {report_path}")
    print(f"Saved confusion matrix: {cm_path}")


save_report_and_confusion(
    true_rating,
    threshold_pred,
    "coral_threshold",
)

save_report_and_confusion(
    true_rating,
    expected_pred,
    "coral_expected",
)


# Borderline ratings 4-7 analysis

borderline_mask = np.isin(true_rating, [4, 5, 6, 7])

borderline_metrics = pd.DataFrame([
    {
        "model": "CORAL-style Threshold",
        "subset": "ratings_4_to_7",
        "n_examples": int(borderline_mask.sum()),
        **compute_basic_rating_metrics(
            true_rating[borderline_mask],
            threshold_pred[borderline_mask],
            "borderline",
        ),
    },
    {
        "model": "CORAL-style Expected",
        "subset": "ratings_4_to_7",
        "n_examples": int(borderline_mask.sum()),
        **compute_basic_rating_metrics(
            true_rating[borderline_mask],
            expected_pred[borderline_mask],
            "borderline",
        ),
    },
])

borderline_metrics = borderline_metrics.rename(columns={
    "borderline_accuracy": "accuracy",
    "borderline_macro_f1": "macro_f1",
    "borderline_weighted_f1": "weighted_f1",
    "borderline_mae": "mae",
    "borderline_within_1": "within_1",
    "borderline_within_2": "within_2",
    "borderline_large_error_ge_3": "large_error_ge_3",
})

borderline_path = CORAL_DIR / "coral_borderline_ratings_4_to_7_metrics.csv"
borderline_metrics.to_csv(borderline_path, index=False)

display(borderline_metrics)
print("Saved borderline metrics to:", borderline_path)